# Anime review sentiment, checked against the reviewers' own verdicts

Runs the whole analysis on your copy of the scraped MyAnimeList reviews. Run the cells in order; each one can be run again safely.

**For the transformer model, turn on a GPU first:** *Runtime → Change runtime type → T4 GPU*. Without one, the notebook skips that model and runs everything else.

In [ ]:
# 1. Get the code (safe to re-run)
import os
os.chdir("/content")
if os.path.isdir("/content/repo/.git"):
    !git -C /content/repo pull -q
else:
    !git clone -q https://github.com/harsh-github007/Anime-Reviews-Sentimental-Analysis.git /content/repo
%cd -q /content/repo
!pip install -q -e ".[model]"
print("Ready in", os.getcwd())

In [ ]:
# 2. Find the data: from Google Drive, or upload it if it isn't there
%cd -q /content/repo
import os
from google.colab import drive
drive.mount("/content/drive")
DATA = "/content/drive/My Drive/Project/Sentiment Analysis of Anime Reviews/Data/Anime Reviews.xlsx"
if not os.path.exists(DATA):
    from google.colab import files
    print("Not found in Drive. Choose the reviews file (.xlsx or .csv) to upload:")
    up = files.upload()
    os.makedirs("data", exist_ok=True)
    DATA = os.path.join("data", next(iter(up)))
    os.replace(next(iter(up)), DATA)
print("Using", DATA)

In [ ]:
# 3. Score every review and check each method against the verdicts (about 20 minutes with a GPU)
%cd -q /content/repo
import torch
flag = "--transformer" if torch.cuda.is_available() else ""
if not flag:
    print("No GPU: skipping the transformer model. Turn on a GPU to include it.")
!python -m animesent --data "$DATA" $flag

In [ ]:
# 4. Show the results
%cd -q /content/repo
import os
from IPython.display import Markdown, display
if os.path.exists("results/results.md"):
    display(Markdown(open("results/results.md").read().replace("figures/", "results/figures/")))
else:
    print("No results yet. Fix the message from step 3, then run it again.")

In [ ]:
# 5. Download the results to add to the repository (charts and numbers only, no review text)
%cd -q /content/repo
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")